# FinTrustX: Deep Learning for Credit Risk

**Home Credit Default Risk | 04_DL_Model**  
**Author:** FinTrustX Data Science Team | **Date:** August 2026

This notebook trains, tunes, evaluates, and persists a reusable feed-forward neural network for binary credit-risk prediction. Model architecture, training orchestration, evaluation, and Optuna logic remain in `src/` modules.

## Objectives

- Load the processed modelling data produced in `02_Preprocessing.ipynb`.
- Establish a configured neural-network baseline on a development split.
- Tune only against validation ROC-AUC, then train the final candidate with early stopping.
- Evaluate the final candidate once on the untouched held-out test set and save the model artifacts.

> **Validation-data note:** the current preprocessing pipeline persists `processed_train.parquet` and `processed_test.parquet`, but not a separate validation artifact. This notebook therefore creates a reproducible stratified validation split from `processed_train.parquet` only. The persisted test set is never used for fitting, tuning, or model selection. Persisting this split upstream would be a useful future pipeline enhancement.

# 1. Environment Check

Configure paths, deterministic seeds, logging, and display settings. The working-directory adjustment keeps the notebook runnable both from the project root and from `notebooks/`.

In [3]:
from __future__ import annotations

import importlib.util
import logging
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split

package_names = {'TensorFlow': 'tensorflow', 'NumPy': 'numpy', 'Pandas': 'pandas', 'scikit-learn': 'scikit-learn', 'Optuna': 'optuna'}
runtime_versions = {'Python': platform.python_version()}
for label, package_name in package_names.items():
    try:
        runtime_versions[label] = version(package_name)
    except PackageNotFoundError:
        runtime_versions[label] = 'MISSING'
display(pd.Series(runtime_versions, name='Version').to_frame())
print(f'Current working directory: {Path.cwd().resolve()}')
try:
    import tensorflow as tf
    print('GPU available' if tf.config.list_physical_devices("GPU") else 'Running on CPU')
except ImportError:
    tf = None
    print('TensorFlow is missing; GPU availability cannot be checked yet. Run the optional installation cell, then restart the kernel.')
if importlib.util.find_spec('google.colab') is None:
    print('WARNING: This notebook may perform heavy computation locally. Connect the notebook to a Google Colab runtime for training.')

def find_project_root(anchor_dirs=('src', 'data', 'models'), max_upward=4) -> Path:
    current = Path.cwd().resolve()
    if (current / 'Credit-risk-ai' / 'src').is_dir():
        return (current / 'Credit-risk-ai').resolve()
    for _ in range(max_upward):
        if all((current / marker).exists() for marker in anchor_dirs):
            return current
        if current.parent == current:
            break
        current = current.parent
    if (Path.cwd() / 'src').exists():
        return Path.cwd().resolve()
    if (Path.cwd().parent / 'src').exists():
        return Path.cwd().parent.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEEP_LEARNING_TRAINING, EARLY_STOPPING, MODEL_DIR, RANDOM_STATE, REPORT_DIR, STUDY_DIR
if tf is not None:
    try:
        from src.evaluate import EvaluationEngine, evaluate_model
        from src.models.neural_network import NeuralNetworkModel
        from src.training.train_dl import DeepLearningTrainer
        from src.tunning.neural_tuner import NeuralNetworkTuner
        from src.tunning.visualization import OptunaVisualizer
    except ImportError as error:
        raise ImportError('Unable to import src modules. Confirm that the complete repository is mounted at the project root. Original error: ' + str(error)) from error

required_modules = {'numpy': 'numpy', 'pandas': 'pandas', 'scikit-learn': 'sklearn', 'tensorflow': 'tensorflow', 'optuna': 'optuna', 'xgboost': 'xgboost', 'catboost': 'catboost', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'shap': 'shap'}
missing_packages = [package for package, module in required_modules.items() if importlib.util.find_spec(module) is None]
print('Missing packages: ' + ', '.join(missing_packages) if missing_packages else 'All required packages are available.')

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold'})
TARGET_COLUMN = DEEP_LEARNING_TRAINING['target_column']
VALIDATION_SIZE = DEEP_LEARNING_TRAINING['validation_split']
N_TRIALS = 5
print(f'Project root verified: {PROJECT_ROOT}; random state: {RANDOM_STATE}; trials: {N_TRIALS}')


,Version
Python,3.12.13
TensorFlow,2.20.0
NumPy,2.0.2
Pandas,2.2.2
scikit-learn,1.6.1
Optuna,MISSING


Current working directory: /content
Running on CPU


RuntimeError: Notebook is not running from the project root. Missing: /content/src, /content/data, /content/models, /content/reports

## Optional dependency installation

Run this cell only after the environment check reports missing packages; restart the kernel afterward.

In [ ]:
# %pip install tensorflow optuna xgboost catboost shap

# 2. Load Processed Data

`DeepLearningTrainer` owns the project-standard processed-data loading and schema validation. The validation set below is carved out of the processed training partition only.

In [ ]:
data_trainer = DeepLearningTrainer()
X_processed_train, y_processed_train, X_test, y_test = data_trainer.load_data()

X_train, X_valid, y_train, y_valid = train_test_split(
    X_processed_train,
    y_processed_train,
    test_size=VALIDATION_SIZE,
    stratify=y_processed_train,
    random_state=RANDOM_STATE,
)

data_shapes = pd.DataFrame(
    {
        'rows': [len(X_train), len(X_valid), len(X_test)],
        'features': [X_train.shape[1], X_valid.shape[1], X_test.shape[1]],
        'default_rate': [y_train.mean(), y_valid.mean(), y_test.mean()],
    },
    index=['train', 'validation', 'test'],
)
display(data_shapes.assign(default_rate=lambda frame: frame['default_rate'].map('{:.2%}'.format)))
assert X_train.shape[1] == X_valid.shape[1] == X_test.shape[1], 'Feature-width mismatch.'
assert not np.shares_memory(X_valid, X_test), 'Validation and test data must be distinct.'

In [ ]:
target_distribution = pd.DataFrame(
    {
        'train': pd.Series(y_train).value_counts(normalize=True),
        'validation': pd.Series(y_valid).value_counts(normalize=True),
        'test': pd.Series(y_test).value_counts(normalize=True),
    }
).rename_axis(TARGET_COLUMN)
display(target_distribution.style.format('{:.2%}'))

missing_summary = pd.DataFrame(
    {
        'train_missing': np.isnan(X_train).sum(axis=0),
        'validation_missing': np.isnan(X_valid).sum(axis=0),
        'test_missing': np.isnan(X_test).sum(axis=0),
    }
)
unexpected_missing = missing_summary[missing_summary.any(axis=1)]
display(unexpected_missing.head() if not unexpected_missing.empty else pd.DataFrame({'status': ['No missing values found.']}))
assert unexpected_missing.empty, 'Processed features contain unexpected missing values.'
assert np.isfinite(X_train).all() and np.isfinite(X_valid).all() and np.isfinite(X_test).all(), 'Processed features must be finite.'

# 3. Baseline Neural Network

The baseline uses the central deep-learning configuration unchanged. `NeuralNetworkModel` supplies the architecture, while its reusable `train` method performs fitting against the validation data.

In [ ]:
baseline_model = NeuralNetworkModel(
    input_dim=X_train.shape[1],
    hidden_layers=DEEP_LEARNING_TRAINING['hidden_layers'],
    units=DEEP_LEARNING_TRAINING['units'],
    dropout=DEEP_LEARNING_TRAINING['dropout'],
    optimizer=DEEP_LEARNING_TRAINING['optimizer'],
    learning_rate=DEEP_LEARNING_TRAINING['learning_rate'],
    hidden_activation=DEEP_LEARNING_TRAINING['hidden_activation'],
    output_activation=DEEP_LEARNING_TRAINING['output_activation'],
    loss=DEEP_LEARNING_TRAINING['loss'],
    metrics=DEEP_LEARNING_TRAINING['metrics'],
    random_state=RANDOM_STATE,
)
baseline_callbacks = [tf.keras.callbacks.EarlyStopping(**EARLY_STOPPING)]
baseline_history = baseline_model.train(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=DEEP_LEARNING_TRAINING['epochs'],
    batch_size=DEEP_LEARNING_TRAINING['batch_size'],
    callbacks=baseline_callbacks,
    verbose=DEEP_LEARNING_TRAINING['verbose'],
)
baseline_model.summary()

# 4. Baseline Evaluation

The shared `EvaluationEngine` calculates all classification and probability metrics and creates the diagnostic plots. Baseline performance is assessed on validation data only.

In [ ]:
baseline_evaluator = EvaluationEngine(model=baseline_model)
baseline_metrics = evaluate_model(baseline_model, X_valid, y_valid, plot=False)
requested_metrics = [
    'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC', 'Average Precision',
    'Balanced Accuracy', 'Matthews Corrcoef', 'Cohen Kappa', 'Log Loss', 'Brier Score Loss',
]
display(pd.Series({metric: baseline_metrics[metric] for metric in requested_metrics}, name='Baseline validation').to_frame().round(4))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
baseline_evaluator.plot_confusion_matrix(X_valid, y_valid, ax=axes[0], title='Baseline — Validation Confusion Matrix')
baseline_evaluator.plot_roc_curve(X_valid, y_valid, ax=axes[1], title='Baseline — Validation ROC Curve')
baseline_evaluator.plot_precision_recall_curve(X_valid, y_valid, ax=axes[2], title='Baseline — Validation Precision-Recall Curve')
plt.tight_layout()
plt.show()

# 5. Baseline Training History

The history returned by the reusable model wrapper reveals convergence and whether validation performance diverges from training performance.

In [ ]:
baseline_history_frame = pd.DataFrame(baseline_history.history)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
baseline_history_frame[['loss', 'val_loss']].plot(ax=axes[0], linewidth=2)
axes[0].set(title='Baseline Training History — Loss', xlabel='Epoch', ylabel='Binary cross-entropy')
baseline_history_frame[['auc', 'val_auc']].plot(ax=axes[1], linewidth=2)
axes[1].set(title='Baseline Training History — AUC', xlabel='Epoch', ylabel='ROC-AUC', ylim=(0, 1))
plt.tight_layout()
plt.show()

# 6. Hyperparameter Tuning

`NeuralNetworkTuner` encapsulates Optuna, the neural-network search space, early stopping, and validation ROC-AUC objective. Five trials are deliberately small for a fast, reproducible demonstration; increase this only after confirming resource availability.

In [ ]:
neural_tuner = NeuralNetworkTuner(random_state=RANDOM_STATE)
neural_study = neural_tuner.optimize(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    n_trials=N_TRIALS,
)

best_trial = neural_study.best_trial
best_params = neural_tuner.get_best_params()
display(pd.DataFrame([{
    'best_trial': best_trial.number,
    'best_validation_roc_auc': neural_study.best_value,
    'best_parameters': best_params,
}]))
display(pd.Series(best_params, name='Best parameter').to_frame())

# 7. Optuna Visualization

The existing `OptunaVisualizer` provides the interactive diagnostics. It displays and saves the requested figures under the configured project reports directory.

In [ ]:
optuna_visualizer = OptunaVisualizer(output_dir=REPORT_DIR / 'optuna' / 'neural_network')
optuna_figures = {
    'optimization_history': optuna_visualizer.optimization_history(neural_study),
    'parameter_importance': optuna_visualizer.parameter_importance(neural_study),
    'parallel_coordinate': optuna_visualizer.parallel_coordinate(neural_study),
    'contour': optuna_visualizer.contour_plot(neural_study),
    'slice': optuna_visualizer.slice_plot(neural_study),
    'edf': optuna_visualizer.edf_plot(neural_study),
}
for figure in optuna_figures.values():
    figure.show()

optuna_artifacts = optuna_visualizer.save_all_plots(neural_study)
display(pd.Series(optuna_artifacts, name='Saved path').to_frame())

# 8. Final Neural Network

The final model is reconstructed with the selected validation parameters and fitted on the same development training split with early stopping. The test partition remains untouched at this stage.

In [ ]:
final_model = NeuralNetworkModel(
    input_dim=X_train.shape[1],
    hidden_layers=best_params['layers'],
    units=best_params['units'],
    dropout=best_params['dropout'],
    optimizer=best_params['optimizer'],
    learning_rate=best_params['learning_rate'],
    hidden_activation=DEEP_LEARNING_TRAINING['hidden_activation'],
    output_activation=DEEP_LEARNING_TRAINING['output_activation'],
    loss=DEEP_LEARNING_TRAINING['loss'],
    metrics=DEEP_LEARNING_TRAINING['metrics'],
    random_state=RANDOM_STATE,
)
final_history = final_model.train(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=DEEP_LEARNING_TRAINING['epochs'],
    batch_size=best_params['batch_size'],
    callbacks=[tf.keras.callbacks.EarlyStopping(**EARLY_STOPPING)],
    verbose=DEEP_LEARNING_TRAINING['verbose'],
)
print(f'Final training stopped after {len(final_history.history["loss"])} epoch(s).')

# 9. Final Evaluation

This is the first and only evaluation of the tuned final model on the held-out test set. The table distinguishes validation baseline results from final test results; they are useful for auditability but are not directly comparable because they use different partitions.

In [ ]:
final_evaluator = EvaluationEngine(model=final_model)
final_metrics = evaluate_model(final_model, X_test, y_test, plot=False)
evaluation_comparison = pd.DataFrame([
    {'Model': 'Baseline neural network (validation)', **{metric: baseline_metrics[metric] for metric in requested_metrics}},
    {'Model': 'Tuned neural network (held-out test)', **{metric: final_metrics[metric] for metric in requested_metrics}},
]).set_index('Model')
display(evaluation_comparison.round(4))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
final_evaluator.plot_confusion_matrix(X_test, y_test, ax=axes[0], title='Tuned Model — Test Confusion Matrix')
final_evaluator.plot_roc_curve(X_test, y_test, ax=axes[1], title='Tuned Model — Test ROC Curve')
final_evaluator.plot_precision_recall_curve(X_test, y_test, ax=axes[2], title='Tuned Model — Test Precision-Recall Curve')
plt.tight_layout()
plt.show()

# 10. Model Saving

Persist the native Keras model and the final held-out evaluation artifacts using filenames and directories defined in project configuration.

In [ ]:
final_model_path = MODEL_DIR / DEEP_LEARNING_TRAINING['model_filename']
final_metrics_path = REPORT_DIR / DEEP_LEARNING_TRAINING['metrics_filename']
final_history_path = REPORT_DIR / DEEP_LEARNING_TRAINING['history_filename']

final_model.save(final_model_path)
final_evaluator.save_metrics(metrics=final_metrics, metrics_path=final_metrics_path)
pd.DataFrame(final_history.history).to_csv(final_history_path, index=False)

artifact_manifest = pd.DataFrame({
    'artifact': ['neural network model', 'test metrics JSON', 'training history CSV'],
    'path': [final_model_path, final_metrics_path, final_history_path],
})
artifact_manifest['exists'] = artifact_manifest['path'].map(lambda path: Path(path).exists())
assert artifact_manifest['exists'].all(), 'One or more expected artifacts were not saved.'
display(artifact_manifest)
for _, row in artifact_manifest.iterrows():
    print(f"{row['artifact']}: {row['path']}")

# 11. Final Results

This compact scorecard prioritizes discrimination and class-aware metrics for model comparison.

In [ ]:
final_results = pd.DataFrame([
    {'Model': 'Baseline neural network (validation)', **{metric: baseline_metrics[metric] for metric in [
        'ROC AUC', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'Average Precision', 'Balanced Accuracy'
    ]}},
    {'Model': 'Tuned neural network (held-out test)', **{metric: final_metrics[metric] for metric in [
        'ROC AUC', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'Average Precision', 'Balanced Accuracy'
    ]}},
])
display(final_results.round(4))

# 12. Conclusions

The baseline establishes the performance of the centrally configured network on the validation partition. The tuned network is selected solely by validation ROC-AUC and is then evaluated once on the held-out test set. Compare the validation and test values cautiously: they use different samples, so a direct score difference is not a causal estimate of tuning benefit. The Optuna best validation ROC-AUC (`neural_study.best_value`) is the appropriate indicator of whether tuning improved over the baseline validation ROC-AUC (`baseline_metrics['ROC AUC']`).

Important limitations remain: five trials provide only a demonstration-level search; the current preprocessing workflow does not persist a dedicated validation artifact; a 0.5 decision threshold may not match business costs; and discrimination metrics do not establish calibration, fairness, stability, or causal validity. Before deployment, expand tuning with resource-aware trials, persist all partitions upstream, assess calibration and subgroup performance, choose a cost-sensitive threshold, and complete governance review.